In [3]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [4]:

import sys

sys.path.append('../scripts')

In [5]:
import numpy as np
import scanpy as sc
import os
import matplotlib.pyplot as plt
import decoupler as dc
import scipy.sparse as sp
import pandas as pd

from cellina import make_neighbor_perturbation
from utils import set_seed
from train_loo import preprocess_crc, preprocess_merfish, _load_model, split_indices
from counterfactual_analysis import compute_lfc_metrics, compute_rmse, compute_edistance, mixing_index, _to_dense
from configs.adata_crc_config import ADATA_ARGS as ADATA_ARGS_CRC
from configs.adata_merfish_config import ADATA_ARGS as ADATA_ARGS_MERFISH

In [7]:
import cellina
cellina.__version__

'0.7.1'

In [8]:
set_seed(0)

In [9]:
DATASET_NAME = "crc"  # or "merfish"
MODEL_ROOT = "/data2/a330d/data/ood/trained"

In [17]:
CRC_PATHS = [
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_210.h5ad",
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_221.h5ad",
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_231.h5ad",
    #"/data2/a330d/datasets/crc/raw_zenodo/crc_232.h5ad",
    "/data2/a330d/datasets/crc/raw_zenodo/crc_242.h5ad",
    "/data2/a330d/datasets/crc/raw_zenodo/crc_120.h5ad",
]

CRC_HOLDOUTS = [
    "Endothelial",
    "Epithelial",
    "Fibroblast",
    "Myeloid",
    "T_cell",
]

MERFISH_PATHS = [
    #"/data2/a330d/datasets/MERFISH_mouse_brain/C57BL6J-2.036.h5ad",    
    #"/data2/a330d/datasets/MERFISH_mouse_brain/C57BL6J-2.039.h5ad",
    "/data2/a330d/datasets/MERFISH_mouse_brain/C57BL6J-2.041.h5ad",
]

MERFISH_HOLDOUTS = [
    'glutamatergic neuron',
    #'oligodendrocyte',
    #'astrocyte',
    #'GABAergic neuron',
    #'endothelial cell',
]

PATHS = CRC_PATHS if DATASET_NAME == "crc" else MERFISH_PATHS
HOLDOUT_CELLTYPES = CRC_HOLDOUTS if DATASET_NAME == "crc" else MERFISH_HOLDOUTS
DATA_ARGS = ADATA_ARGS_CRC if DATASET_NAME == "crc" else ADATA_ARGS_MERFISH

In [18]:
n_top_genes = DATA_ARGS.get('n_top_genes')
labels_key = DATA_ARGS.get('labels_key')
domains_key = DATA_ARGS.get('domains_key')
batch_key = DATA_ARGS.get('batch_key')
control_domain = DATA_ARGS.get('control_domains')[0]
holdout_domains = DATA_ARGS.get('holdout_domains')
n_neighbors = DATA_ARGS.get('n_neighbors')
batch_size = 2048
library_size = 1e4
n_deg = 50
n_pert_genes = 200

In [19]:
# Create SLIDES which contain file names from PATHS - first split by "/" and take last part, then split by "." and take first part
SLIDES = [path.split("/")[-1].split(".")[0] for path in PATHS]

In [20]:
def get_perturbation_logfc(adata, control_domain, holdout_domain, labels_key, domains_key):
    # Cell-type-specific
    pdata_ct = dc.pp.pseudobulk(
        adata=adata, sample_col=domains_key, groups_col=labels_key, mode='sum', layer='counts'
    )
    sc.pp.normalize_total(pdata_ct, target_sum=1e4)
    sc.pp.log1p(pdata_ct)

    cell_types_with_both = [
        ct for ct in pdata_ct.obs[labels_key].unique()
        if ((pdata_ct.obs[domains_key] == control_domain) & (pdata_ct.obs[labels_key] == ct)).any()
        and ((pdata_ct.obs[domains_key] == holdout_domain) & (pdata_ct.obs[labels_key] == ct)).any()
    ]

    _ct_rows = []
    for _ct in cell_types_with_both:
        _crc_ct = pdata_ct[(pdata_ct.obs[domains_key] == holdout_domain) & (pdata_ct.obs[labels_key] == _ct)].X
        _ref_ct = pdata_ct[(pdata_ct.obs[domains_key] == control_domain) & (pdata_ct.obs[labels_key] == _ct)].X
        _crc_m  = np.asarray(_crc_ct.mean(axis=0)).flatten() if sp.issparse(_crc_ct) else _crc_ct.mean(axis=0).flatten()
        _ref_m  = np.asarray(_ref_ct.mean(axis=0)).flatten() if sp.issparse(_ref_ct) else _ref_ct.mean(axis=0).flatten()
        _ct_rows.append(pd.Series(_crc_m - _ref_m, index=pdata_ct.var_names, name=_ct))
    domain_logfc_df = pd.concat(_ct_rows, axis=1).T

    return domain_logfc_df

In [ ]:
results = []
for path, slide_id in zip(PATHS, SLIDES):
    adata = sc.read(path)
    if DATASET_NAME == 'crc':
        adata = preprocess_crc(adata, n_top_genes=n_top_genes, n_neighbors=n_neighbors, labels_key=labels_key, domains_key=domains_key)
    elif DATASET_NAME == 'merfish':
        adata = preprocess_merfish(adata, n_top_genes=n_top_genes, n_neighbors=n_neighbors, labels_key=labels_key, domains_key=domains_key)
    else:
        raise ValueError(f"Unknown dataset_name: {DATASET_NAME}. Supported: crc, merfish")
    

    for holdout_celltype in HOLDOUT_CELLTYPES:
        # 50 times * in print
        print(f"{'='*50} Slide: {slide_id}, Holdout Celltype: {holdout_celltype} {'='*50}")
        # create splits
        train_idx, val_idx, test_idx = split_indices(adata,
                                                    holdout_celltype,
                                                    labels_key=labels_key,
                                                    domains_key=domains_key,
                                                    holdout_domains=holdout_domains,
                                                    seed=0)

        splits = (train_idx, val_idx, test_idx)

        save_dir = os.path.join(MODEL_ROOT, slide_id, holdout_celltype, 'cellina')
        model = _load_model(save_dir,
                            model_class='cellina',
                            adata=adata,
                            splits=splits)
        is_control_region = adata.obs[domains_key]==(control_domain)
        is_holdout_ct = adata.obs[labels_key].astype(str) == holdout_celltype
        mask_control = is_control_region & is_holdout_ct
        idx_control = np.where(mask_control.values)[0]    

        for hd in holdout_domains:
                # ── Cell-type-specific perturbation ──────────────────────────────────────
                domain_logfc_df = get_perturbation_logfc(adata, control_domain, hd, labels_key, domains_key)
                logfc_series_dict = {}
                s = domain_logfc_df.loc[holdout_celltype]
                logfc_series_dict[holdout_celltype] = s[s.abs().nlargest(n_pert_genes).index]
                make_neighbor_perturbation(
                        adata, perturbations=logfc_series_dict, groupby=labels_key,
                        obsm_key_out='spatial_x_cf', base=np.e,
                )
                pert_expr = model.get_perturbed_expression(
                        adata=adata, indices=idx_control, spatial_obsm_key='spatial_x_cf',
                        batch_size=batch_size, library_size=library_size,
                        )
                
                is_holdout_region = adata.obs[domains_key].astype(str) == hd
                mask_target = is_holdout_region & is_holdout_ct
                idx_target = np.where(mask_target.values)[0]
                
                # Compute stats
                control = adata.layers['counts'][mask_control.values, :]
                target = adata.layers['counts'][mask_target.values, :]
                control, target = _to_dense(control), _to_dense(target)
                counterfactual = pert_expr

                pear, spear, prec, dir_match, deg = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg)
                rmse = compute_rmse(observed=target, predicted=counterfactual, deg=deg)
                edist_global = compute_edistance(observed=target, predicted=counterfactual, deg=deg)
                edist_local = compute_edistance(observed=target, predicted=counterfactual, deg=deg, local=True)
                mix_idx = mixing_index(observed=target, predicted=counterfactual)
                _, _, _, dir_match_k, _ = compute_lfc_metrics(control=control, target=target, counterfactual=counterfactual, n_deg=n_deg, direction_match_normalize="k")

                results.append(
                        dict(
                        dataset_name=DATASET_NAME,
                        sid=slide_id,
                        control_domain=control_domain,
                        target_domain=hd,
                        n_deg=n_deg,
                        model_name="cellina-pert",
                        holdout_celltype=holdout_celltype,
                        spearman=spear,
                        pearson=pear,
                        precision=prec,
                        direction_match=dir_match,
                        direction_match_k=dir_match_k,
                        mixing_index=mix_idx,
                        edistance_global=edist_global,
                        edistance_local=edist_local,
                        rmse=rmse,
                        top_n_perturb=n_pert_genes,
                        )
                )

In [ ]:
results

In [23]:
results_csv_name = f'../results/loo_cellina_{DATASET_NAME}_DEG_{n_deg}'
results_csv_path = results_csv_name + '_pert.csv'
df_results = pd.DataFrame(results)
if os.path.exists(results_csv_path):
    df_results.to_csv(f"{results_csv_path}", mode='a', header=False, index=False)
else:
    df_results.to_csv(f"{results_csv_path}", index=False)